# Gradio

In this notebook, We will build User Interfaces using the outrageously simple Gradio framework.


Please note: your Gradio screens may appear in 'dark mode' or 'light mode' depending on your computer settings.

## Imports and API Call Setting:

In [12]:
# Imports:

import os
from dotenv import load_dotenv
from openai import OpenAI

import gradio as gr

In [13]:
# Loading Environment Variables for API call to OpenAI:

load_dotenv(override= True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print('OpenAI API Key Found!')
else:
    print('OpenAI API Key Not Found!')

OpenAI API Key Found!


In [14]:
# Connecting to OpenAI:

openai = OpenAI()

In [15]:
# Simple Call to GPT-4.1-mini to Test Connection:

system_message = 'You are a helpful assistant.'

def message_gpt(prompt):
    messages = [{'role': 'system', 'content': system_message},
                {'role': 'user', 'content': prompt}]

    response = openai.chat.completions.create(model= 'gpt-4.1-mini', messages= messages)

    return response.choices[0].message.content

In [16]:
# This can reveal the "training cut off", or the most recent date in the training data

message_gpt("What is today's date?")

"Today's date is April 27, 2024."

## User Interface using Gradio:

In [17]:
# Simple Function:

def shout(text):
    """
    Turns given text to all capitals.
    :param text: Text to shout.
    :return: Text in All capital letters.
    """

    print(f'Shout has been called with input {text}')
    return text.upper()

In [18]:
shout('hello')

Shout has been called with input hello


'HELLO'

In [19]:
gr.Interface(fn= shout,
             inputs= 'textbox',
             outputs= 'textbox',
             flagging_mode= 'never').launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [20]:
# Adding share=True means that it can be accessed publicly.

# A more permanent hosting is available using a platform called Spaces from HuggingFace, which we will touch on next week

# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True.
# If you're at work on a work network, I suggest skip this test.

gr.Interface(fn= shout, inputs= 'textbox', outputs= 'textbox', flagging_mode= 'never').launch(share=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://d6a3820ac39f59841a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
# Adding inbrowser=True opens up a new browser window automatically

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Adding authentication

Gradio makes it very easy to have userids and passwords.

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env.

In [22]:
gr.Interface(fn= shout,
             inputs= 'textbox',
             outputs= 'textbox',
             flagging_mode="never").launch(inbrowser=True,
                                           auth= ('shail', 'baby dont hurt me'))

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


## Forcing Dark / Light Mode:

Gradio appears in light mode or dark mode depending on the settings of the browser and computer.

There is a way to force gradio to appear in dark mode, but Gradio recommends against this as it should be a user preference (particularly for accessibility reasons).
But if you wish to force dark mode for your screens, below is how to do it.

In [23]:
# Define this variable and then pass js=force_dark_mode when creating the Interface

# Change 'light' to 'dark' to force Dark Mode:
force_light_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'light') {
        url.searchParams.set('__theme', 'light');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs= 'textbox', outputs= 'textbox', flagging_mode= 'never').launch(js= force_light_mode)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## Adding a little more to web-page:

In [24]:
message_input = gr.Textbox(label= 'Your Message:',
                           info= 'Enter a message to be shouted',
                           lines= 2)

message_output = gr.Textbox(label= 'Response:',
                            lines= 2)

view = gr.Interface(
    fn= shout,
    title= 'Shout',
    inputs= [message_input],
    outputs= [message_output],
    examples= ['Abey oye', 'Ruk bey'],
    flagging_mode= 'never'
)

view.launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [25]:
# And now - changing the function from "shout" to "message_gpt":

message_input = gr.Textbox(label= 'Your Message:',
                           info= 'Enter a message for GPT-4.1-Mini',
                           lines= 5)

message_output = gr.Textbox(label= 'Response:',
                            lines= 10)

view = gr.Interface(
    fn= message_gpt,
    title= 'GPT-4.1',
    inputs= [message_input],
    outputs= [message_output],
    examples= ['Hii', 'Hello'],
    flagging_mode= 'never'
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## Using Markdown Response:

In [26]:
# Redefining System message:

system_message = 'You are a helpful assistant that responds in markdown without code blocks.'

message_input = gr.Textbox(label= 'Your Message:',
                           info= 'Enter a message for GPT-4.1-Mini',
                           lines= 5)

message_output = gr.Markdown(label= 'Response:')

view = gr.Interface(
    fn= message_gpt,
    title= 'GPT-4.1',
    inputs= [message_input],
    outputs= [message_output],
    examples= [
        'Explain the Transformer architecture to a layperson', 'Explain the Transformer architecture to an aspiring AI engineer'],
    flagging_mode= 'never'
)

view.launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


## Streaming Results:

In [27]:
# Creating a Function that Streams Back Results:

def stream_gpt(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': "user", 'content': prompt}
      ]
    stream = openai.chat.completions.create(
        model= 'gpt-4.1-mini',
        messages= messages,
        stream= True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [28]:
message_input = gr.Textbox(label= 'Your Message:',
                           info= 'Enter a message for GPT-4.1-Mini',
                           lines= 5)

message_output = gr.Markdown(label= 'Response:')

view = gr.Interface(
    fn= stream_gpt,
    title= 'GPT-4.1',
    inputs= [message_input],
    outputs= [message_output],
    examples= [
        'Explain the Transformer architecture to a layperson', 'Explain the Transformer architecture to an aspiring AI engineer'],
    flagging_mode= 'never'
)

view.launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


## Streaming Results using Ollama:

In [29]:
ollama_url = 'http://172.23.37.63:11434/v1'

ollama = OpenAI(base_url= ollama_url,
                api_key= '')

# Creating a Function that Streams Back Results from OLLAMA:

def stream_ollama(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': "user", 'content': prompt}
      ]
    stream = ollama.chat.completions.create(
        model= 'llama3.1:8b',
        messages= messages,
        stream= True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [30]:
message_input = gr.Textbox(label='Your Message:',
                           info='Enter a message for LLAMA3.1:8b',
                           lines=5)

message_output = gr.Markdown(label='Response:')

view = gr.Interface(
    fn= stream_ollama,
    title='Ollama',
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        'Explain the Transformer architecture to a layperson',
        'Explain the Transformer architecture to an aspiring AI engineer'],
    flagging_mode='never'
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## Multi-Model UI:

In [31]:
def stream_model(prompt, model):

    if model == 'GPT':
        result = stream_gpt(prompt)

    elif model == 'OLLAMA':
        result = stream_ollama(prompt)

    else:
        raise ValueError('Invalid model')

    yield from result

In [32]:
message_input = gr.Textbox(label= 'Your Message:',
                           info= 'Enter a message for LLM',
                           lines= 5)

model_selector = gr.Dropdown(choices= ['GPT', 'OLLAMA'], label= 'Select Model', value= 'GPT')

message_output = gr.Textbox(label = 'Response:')

view = gr.Interface(
    fn= stream_model,
    title= 'LLMs',
    inputs= [message_input, model_selector],
    outputs= [message_output],
    examples= [
        ['Explain the Transformer architecture to a layperson', 'GPT'],
        ['Explain the Transformer architecture to an aspiring AI engineer' , 'OLLAMA'],
    ],
    flagging_mode= 'never'
)

view.launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.
